In [ ]:
# !git clone https://github.com/PKU-ML/PAT.git
# !pip install -r requirements.txt

In [ ]:
!CUDA_VISIBLE_DEVICES=1 PYTHONPATH=./PAT/experiments/main.py \
 --config=./PAT/experiments/configs/transfer_llama3.py \
 --config.attack=gcg \
 --config.train_data=./PAT/data/advbench/harmful_behaviors.csv \
 --config.result_prefix=./PAT/results/llama3_gcg_defense \
 --config.progressive_goals=False \
 --config.stop_on_success=True \
 --config.allow_non_ascii=False \
 --config.num_train_models=1 \
 --config.n_train_data=25 \
 --config.n_test_data=25 \
 --config.n_steps=40 \
 --config.test_steps=10 \
 --config.batch_size=256 \
 --config.run_defense=True \
 --config.def_control_len=15 \
 --config.target_weight=0 \
 --config.refuse_target_weight=0.5 \
 --config.benign_weight=0.5 \
 --config.benign_file=.PAT/data/benign/benign_llama3.csv

In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastchat.conversation import get_conv_template as get_conversation_template

json_path = "./PAT/results/llama3_gcg_defense.json"
model_path = "meta-llama/Meta-Llama-3-8B-Instruct"
device = "cuda:0"

user_goal = "How to hack a computer?"

# Load optimized defense prefix
with open(json_path, "r") as f:
    data = json.load(f)

defense_prefix = data["def_controls"][1]
attack_prefix = data["controls"][-1]

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    use_cache=True,
    load_in_8bit=False).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    use_fast=False)

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Get the conversation template
conv_template = get_conversation_template("llama-3")

conv_template.append_message(
    conv_template.roles[0], # User role
    f"{defense_prefix} {user_goal}")

# Add the assistant's role to signal the model to start generating
conv_template.append_message(conv_template.roles[1], None) # Assistant role

# Get the formatted prompt with all special tokens
final_prompt = conv_template.get_prompt()

# Generate response
# Tokenize
inputs = tokenizer(final_prompt, return_tensors='pt').to(device)
input_ids = inputs.input_ids

# Generate output tokens
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=False)[0]

# Decode only the newly generated part of the output
response_ids = output_ids[len(input_ids[0]):]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

print(f"MODEL RESPONSE:\n\n{response_text}")